## Workshop Lab 4: Dynamic Symbology and Labeling with Arcade

### What is Arcade?
Up until now, you've used Python and Pandas to clean and structure your data outside of ArcGIS Pro. But inside ArcGIS Pro, [Arcade](https://developers.arcgis.com/arcade/) is Esri’s lightweight expression language designed specifically for real-time calculations, custom labeling, and advanced symbology on the fly. That is, without modifying your underlying attribute table.

### Symbolizing Roadrunner Observations by Season
Instead of manually symbolizing or splitting our data into multiple feature classes, we can use Arcade to symbolize. For example, we can use an expression to determine the season of a given observation and then color code by season.

#### Steps in ArcPro:
- In ArcPro, right click on `all_roadrunners_cleaned_XYtoPoint` (feature class), then click on Symbology.
- Change symbology type to Unique Values
- Open Arcade Expression editor (green X button)
- Erase the default expression
- Use the Arcade from the chunk below to symbolize by season (paste the code below into the arcade editor in ArcPro):

In [ ]:
var m = Month($feature.date_dt);
// Dec, Jan, Feb -> Winter
if (m == 11 || m == 0 || m == 1) { return "Winter"; }
// Mar, Apr, May -> Spring
else if (m >= 2 && m <= 4) { return "Spring"; }
// Jun, Jul, Aug -> Summer
else if (m >= 5 && m <= 7) { return "Summer"; }
// Sep, Oct, Nov -> Fall
else { return "Fall"; }

- Click 'Run' or the green checkmark to check for syntax errors
- Title the expression (e.g. observations by season)
- Apply (Click Ok)

Question:
- Which months are considered part of the "Winter" category?

### Custom Labeling with Dynamic Strings
Arcade can also create clean map labels that combine multiple fields. Let's practice using our `all_roadrunners_cleaned_XYtoPoint` feature class.

#### Steps in ArcPro:
- Navigate to the Labeling tab or right click the data layer and select Labeling Properties
- Change the **Class Field** by clicking on the green expression button.
- Write Arcade to label by common_name and observation year:

In [ ]:
// Combine common name and year into a clean label string
return $feature.common_name + " (" + $feature.obs_year + ")";

Make sure you have turned labels on (right click the layer, click on labels).

### Challenge Exercise:

For `all_roadrunners_cleaned_XYtoPoint`, write an Arcade expression that calculates how many years have passed since an observation occurred. 
- Do this in the symbology, unique values expression editor. 
- Call this `AgeOfObs`.

<details>
<summary><b>Click here for a hint!</b></summary>

Hint: Use `Year(Today()) - $feature.obs_year` to symbolize by record age.
</details>

In the symbology pane, click on `more` (upper right corner of symbols table), then `show count`. How many observations are missing a date and how can you tell?

<details>
<summary><b>Click here for answer</b></summary>

There are 41 observations with an AgeOfObs value of 2026. This suggests that the obs_year column has no data for those 41 observations.
</details>

Open the attribute table for the data layer. Is there a new AgeOfObs column?

### Challenge Part 2: Cursor Editing

As you saw above, using Arcade to symbolize by AgeOfObs does **not** add a new AgeOfObs column. 

We can use python to add this as a new column. We will do this inside ArcPro!
- In ArcPro, go to the Analysis tab, then click Python. This opens a Jupyter notebook inside the project.
- Copy the chunk below (note that the fc path is just our feature class! This is because arcpy will look inside the open project automatically)
- Paste the code chunk into the notebook and press ctrl+Enter to run.

In [ ]:
# Python equivalent to the Arcade symbology - running this WILL add a new column to your attribute table.

import arcpy
from datetime import date

# Use the layer name exactly as it appears in the Contents pane
fc = "all_roadrunners_cleaned_XYtoPoint"
this_year = date.today().year

# 1) Make an empty column to hold the result
arcpy.management.AddField(fc, "AgeOfObs_Test", "LONG")

# 2) Walk the table row by row and fill it in
with arcpy.da.UpdateCursor(fc, ["obs_year", "AgeOfObs_Test"]) as cursor:
    for row in cursor:                  # row is a list: row[0] = obs_year, row[1] = AgeOfObs
        if row[0] is not None:          # skip observations with no date
            row[1] = this_year - row[0]
            cursor.updateRow(row)       # without this line, the change is never saved

Now, if you check your attribute table you should see an AgeOfObs_Test column!

### Pop ups
Let's try another expression. This time we want to know how many roadrunners were observed within 500 m of coyotes. We will do this using popups on the coyotes!
- Add the `cleaned_coyotes_data_XYtoPoint` feature class to the map (check the `backup` geodatabase if you didn't create this in notebook 3).
- Right click on the layer, click symbology, and change the symbol to something easy to read (e.g. a large red circle). This will make it easier to answer the questions below.
- Right click on the layer and select `Configure Popups`
- Click on Expressions button at the bottom of the panel. Then click new.
- Copy the code below into the editor. Name it `Roadrunners500m` and then click OK:


In [ ]:
// Get the Roadrunner layer from the map context
var roadrunners = FeatureSetByName($map, "all_roadrunners_cleaned_XYtoPoint");

// Create a true 500-meter geodesic buffer around the clicked coyote point
var search_area = BufferGeodetic($feature, 500, 'meters');

// Find how many Roadrunner points fall inside that buffer
var nearby = Intersects(roadrunners, search_area);
var count_nearby = Count(nearby);

// Return a clean dynamic string for the popup
if (count_nearby > 0) {
    return "⚠️ " + count_nearby + " Greater Roadrunner(s) recorded within 500m of this coyote.";
} else {
    return "✅ No roadrunners recorded within 500m.";
}

- Click on the back arrow (upper left of pane) to go back to main view
- Click on `Text` at the top to add a text block
- Click on `Field` and select your expression (`Roadrunners500m`)
- Add any other text you'd like!
- Click anywhere outside the pane to save the expression
- Use the Explore tool to test the pop up

Question: For the coyote observation farthest east in Oklahoma (id 246772934), what does the pop up say for number of roadrunners?

<details>
<summary><b>Click here for answer</b></summary>

No roadrunners recorded within 500m.
</details>

Zoom in on the Albuquerque area and look at the University of New Mexico campus (University and Central). For the coyote on campus (observation id 131680323), what year was it observed? And how many roadrunners does the pop up return?

<details>
<summary><b>Click here for answer</b></summary>

This coyote was observed in 2022 and there are 510 Great Roadrunner(s) recorded within 500m of this coyote.
</details>

### Let's add some fun symbols!
- In the contents pane, left click on the symbol for coyote to open the symbology pane.
- Switch to the properties tab. Click on the Layers icon.
- Switch the Layer Type to Picture.
- Browse to 02_notebooks -> symbols and select the charming-cartoon-coyote... file
- Adjust Size (e.g. 24 pt)

### Now try the same for roadrunners!
Use the roadrunner-cartoon-character... file.

Note: This might be too much for your computer to handle. If so, try zooming in on a small area to see if you can get the icons to load.


### Challenge:
Let's symbolize coyote observations by scale based on age of observation.

- Right click on the coyotes layer, open the symbology pane, keep the single symbol. This should still be set as the cartoon png.
- Click on 'Vary Symbology by Attribute' (green squares)
- Click on the Size dropdown. Then the expression editor (green X)
- Enter the expression below:


In [ ]:
// Years relative to today: 0 = this year, negative = older. Recent observations get the biggest symbols.
return $feature.obs_year - Year(Today());

- Check the "Enable size range" box.
- Check the "Show legend" box.

The coyote symbol is now scaled to match the age of the observation, with recent observations being larger and older observations being smaller!

### Why Arcade?
So why is this notebook dedicated to Arcade? As an ESRI specific language, Arcade is useful across ESRI products. It is a powerful way to visualize as needed without changing the underlying data. This means you can symbolize data without increasing file sizes or calculating new values.

You've now used Arcade for four major GIS tasks:
1. **Dynamic Symbology** (visualizing by season)
2. **Custom Labeling** (string concatenation on species and year)
3. **Spatial Analysis in Popups** (`Intersects` and `BufferGeodetic` across two different layers)
4. **Data-Driven Variable Sizing** (scaling symbols by age of observation)

The beauty of Arcade is that **none of these calculations permanently altered your CSV or feature class attributes**. Instead, they were calculated entirely on the fly by ArcGIS Pro's rendering engine. If your underlying data updates tomorrow, your symbols, labels, and popups update automatically!

#### What else can Arcade do?
1) **Geometry Functions** (Distance Calculations):
Instead of just checking counts within a buffer for a popup (our Roadrunners500m example), you can calculate the exact distance from a point to a landmark or feature dynamically. For example, adding a popup line that reads: "This observation is 1.4 miles away from the nearest highway."
2) **HTML Formatting in Popups**:
Arcade can return HTML tags. You can make popups use bold text, colored warning emojis, or even embed dynamic web links/images based on attribute values.
3) **Advanced Conditional Logic and Custom Binning**:
Instead of just classifying by season or age, you can build multi-conditional risk or activity tiers (e.g., combining time of day and season to categorize observations into "Peak Activity vs. Off-Peak").

## Next Step: Open 05_SharingAndAPIs.ipynb